In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from tqdm.notebook import tqdm
from sklearn.preprocessing import StandardScaler
import joblib
from sklearn.metrics import classification_report, confusion_matrix
import copy
import seaborn as sns


In [22]:
df0404 = pd.read_csv("../data/labeledAndAnomaly/no_plug/0404.csv")
df0712 = pd.read_csv("../data/labeledAndAnomaly/no_plug/0712.csv")
df1703 = pd.read_csv("../data/labeledAndAnomaly/no_plug/1703.csv")
df2906_1 = pd.read_csv("../data/labeledAndAnomaly/no_plug/2906-1.csv")
df3006_1 = pd.read_csv("../data/labeledAndAnomaly/no_plug/3006-1.csv")
df3006_2 = pd.read_csv("../data/labeledAndAnomaly/no_plug/3006-2.csv")

df1103 = pd.read_csv("../data/labeledAndAnomaly/partial_plug/1103.csv")
df1803 = pd.read_csv("../data/labeledAndAnomaly/partial_plug/1803.csv")
df2503 = pd.read_csv("../data/labeledAndAnomaly/partial_plug/2503.csv")
df2108_1 = pd.read_csv("../data/labeledAndAnomaly/partial_plug/2108-1.csv")


df0612_2 = pd.read_csv("../data/labeledAndAnomaly/plug/0612-2.csv")
df0612_3 = pd.read_csv("../data/labeledAndAnomaly/plug/0612-3.csv")
df0707_1 = pd.read_csv("../data/labeledAndAnomaly/plug/0707-1.csv")
df0707_2 = pd.read_csv("../data/labeledAndAnomaly/plug/0707-2.csv")
df0707_4 = pd.read_csv("../data/labeledAndAnomaly/plug/0707-4.csv")
df0709 = pd.read_csv("../data/labeledAndAnomaly/plug/0709.csv")
df1003 = pd.read_csv("../data/labeledAndAnomaly/plug/1003.csv")
df1112 = pd.read_csv("../data/labeledAndAnomaly/plug/1112.csv")
df1310_2 = pd.read_csv("../data/labeledAndAnomaly/plug/1310-2.csv")
df1310_3 = pd.read_csv("../data/labeledAndAnomaly/plug/1310-3.csv")
df1407_1 = pd.read_csv("../data/labeledAndAnomaly/plug/1407-1.csv")
df1407_2 = pd.read_csv("../data/labeledAndAnomaly/plug/1407-2.csv")
df1407_3 = pd.read_csv("../data/labeledAndAnomaly/plug/1407-3.csv")
df1407_4 = pd.read_csv("../data/labeledAndAnomaly/plug/1407-4.csv")
df1503 = pd.read_csv("../data/labeledAndAnomaly/plug/1503.csv")
df1811 = pd.read_csv("../data/labeledAndAnomaly/plug/1811.csv")
df2106_2 = pd.read_csv("../data/labeledAndAnomaly/plug/2106-2.csv")
df2108_2 = pd.read_csv("../data/labeledAndAnomaly/plug/2108-2.csv")
df2701 = pd.read_csv("../data/labeledAndAnomaly/plug/2701.csv")
df2906_2 = pd.read_csv("../data/labeledAndAnomaly/plug/2906-2.csv")
df2906_3 = pd.read_csv("../data/labeledAndAnomaly/plug/2906-3.csv")
df2906_4 = pd.read_csv("../data/labeledAndAnomaly/plug/2906-4.csv")
df3006_3 = pd.read_csv("../data/labeledAndAnomaly/plug/3006-3.csv")

In [23]:
no_plug_dfs = [df0404, df0712, df1703, df2906_1, df3006_1, df3006_2]
plug_dfs =  [df0612_2, df0612_3, df0707_1, df0707_2, df0707_4, df0709, df1003, df1112, df1310_2, df1310_3,
            df1407_1, df1407_2, df1407_3, df1407_4, df1503, df1811, df2106_2, df2108_2, df2701, 
            df2906_2, df2906_3, df2906_4, df3006_3]
partial_plug_dfs = [df1103, df1803, df2503, df2108_1]

In [47]:
"""
Grouped (run-level) cross-validation that mirrors the final model setup.

Mirrors the thesis setup:
  - 4 baseline-relative pressure features + Isolation Forest anomaly score (5 inputs)
  - 60-sample windows, label of the sample right after the window
  - single unidirectional LSTM (32 units), last hidden state, dropout 0.4, linear -> 4 logits
  - inverse-frequency class weights, State 3 weight x1.1
  - AdamW (lr 1e-3, weight decay 1e-3), gradient clipping 1.0
  - ReduceLROnPlateau (x0.5, patience 3), early stopping (patience 10, max 100 epochs)
  - training-only augmentation (gain, offset, noise, channel removal)
  - batch size 256 (train) / 512 (eval)

Differences from a plain grouped CV, on purpose:
  - Isolation Forest + both scalers are refit inside every fold on training runs only.
  - Early stopping uses an inner validation split of TRAINING runs, never the held-out fold.
"""
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)

BASE_FEATURES = ["rel_tot_dp_change", "rel_ts_dp_change",
                 "rel_p_ts_dp_change", "Relative_Ratio_Dev"]
WINDOW = 60
N_CLASSES = 4


# ----------------------------------------------------------------------------
# Model (same architecture as the final model)
# ----------------------------------------------------------------------------
class HydraulicLSTM(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=32, num_classes=4, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=1, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        _, (h, _) = self.lstm(x)       # h: (1, B, H) for a single layer
        return self.fc(self.drop(h[-1]))


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_windows(feat, y, window=WINDOW, within_sections=False):
    """Windows feat[t-window:t] with target y[t]. If within_sections=True, only keep
    windows whose whole span (and target) lies in one health-state section. Labels are
    monotone within a run, so equal endpoint labels imply a single section."""
    n = len(y)
    if n <= window:
        return (np.empty((0, window, feat.shape[1]), np.float32),
                np.empty((0,), np.int64))
    sw = np.lib.stride_tricks.sliding_window_view(feat, window, axis=0)  # (n-w+1, F, w)
    X = np.ascontiguousarray(sw[: n - window].transpose(0, 2, 1), dtype=np.float32)
    t = y[window:].astype(np.int64)
    if within_sections:
        keep = y[: n - window] == y[window:]
        X, t = X[keep], t[keep]
    return X, t


def augment(x, mu, sd, gain_sd=0.3, offset_sd=0.3, noise_sd=0.1, drop_p=0.2):
    """Training-only perturbations on standardized windows x: (B, T, F).
    gain is applied to the unscaled value: x_g = g*x + (g-1)*mu/sd."""
    B, T, F = x.shape
    g = torch.exp(torch.randn(B, 1, F) * gain_sd)
    off = torch.randn(B, 1, F) * offset_sd
    noise = torch.randn(B, T, F) * noise_sd
    x = g * x + (g - 1.0) * (mu / sd) + off + noise
    drop = torch.rand(B, 1, F) < drop_p          # channel removal -> training mean (= 0)
    return torch.where(drop, torch.zeros_like(x), x)


def class_weights(y_train, boost_class3=1.1):
    counts = np.bincount(y_train, minlength=N_CLASSES).astype(float)
    counts[counts == 0] = 1.0
    w = len(y_train) / (N_CLASSES * counts)
    w[3] *= boost_class3
    return torch.tensor(w, dtype=torch.float32)


def fit_isolation_forest(train_runs, train_frac=0.6, seed=42):
    """Fit scaler + Isolation Forest on the first 60% of no-plug TRAINING runs."""
    rows = [r["X4"][: int(train_frac * len(r["X4"]))] for r in train_runs if r["is_no_plug"]]
    if not rows:
        raise ValueError("No no-plug run in the training folds; cannot fit Isolation Forest.")
    rows = np.vstack(rows)
    sc = StandardScaler().fit(rows)
    iso = IsolationForest(n_estimators=300, max_samples=0.75, max_features=1.0,
                          contamination=0.0005, bootstrap=False,
                          random_state=seed).fit(sc.transform(rows))
    return sc, iso


def inner_validation_split(train_runs, frac=0.2, seed=0):
    """Hold out ~20% of the TRAINING runs (per run type) for early stopping."""
    rng = np.random.RandomState(seed)
    groups = {"none": [], "partial": [], "plug": []}
    for r in train_runs:
        groups[r["type"]].append(r)
    tr, va = [], []
    for g in groups.values():
        idx = rng.permutation(len(g))
        n_val = max(1, int(round(frac * len(g)))) if len(g) > 1 else 0
        va += [g[i] for i in idx[:n_val]]
        tr += [g[i] for i in idx[n_val:]]
    return tr, va


def evaluate(model, X, y, mu, sd, weights, device, batch=512):
    model.eval()
    loss_sum, w_sum, preds = 0.0, 0.0, []
    w_dev = weights.to(device)
    crit = nn.CrossEntropyLoss(weight=w_dev, reduction="sum")
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = torch.from_numpy(X[i:i + batch]).to(device)
            xb = (xb - mu.to(device)) / sd.to(device)
            yb = torch.from_numpy(y[i:i + batch]).to(device)
            out = model(xb)
            loss_sum += crit(out, yb).item()
            w_sum += w_dev[yb].sum().item()      # same normalization as PyTorch's weighted mean
            preds.append(out.argmax(1).cpu().numpy())
    return loss_sum / max(w_sum, 1e-12), np.concatenate(preds) if preds else np.array([], int)


def train_one_fold(Xtr, ytr, Xva, yva, n_features, device, seed,
                   max_epochs=100, patience=10, batch=256, verbose=True):
    set_seed(seed)
    flat = Xtr.reshape(-1, n_features)
    mu = torch.tensor(flat.mean(0, dtype=np.float64), dtype=torch.float32).view(1, 1, -1)
    sd = torch.tensor(flat.std(0, dtype=np.float64) + 1e-8, dtype=torch.float32).view(1, 1, -1)
    w = class_weights(ytr)

    model = HydraulicLSTM(input_dim=n_features).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    crit = nn.CrossEntropyLoss(weight=w.to(device))

    Xtr_t = torch.from_numpy(Xtr)
    ytr_t = torch.from_numpy(ytr)
    best_loss, best_state, best_epoch, bad = float("inf"), None, 0, 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), batch):
            idx = perm[i:i + batch]
            xb = (Xtr_t[idx] - mu) / sd
            xb = augment(xb, mu, sd).to(device)
            yb = ytr_t[idx].to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        val_loss, _ = evaluate(model, Xva, yva, mu, sd, w, device)
        sched.step(val_loss)
        if val_loss < best_loss - 1e-6:
            best_loss, best_epoch, bad = val_loss, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
        if verbose and epoch % 5 == 0:
            print(f"    epoch {epoch:3d}  inner-val loss {val_loss:.4f}")
    model.load_state_dict(best_state)
    return model, mu, sd, w, best_epoch


# ----------------------------------------------------------------------------
# Main entry point
# ----------------------------------------------------------------------------
def run_grouped_cv(no_plug_dfs, partial_plug_dfs, plug_dfs,
                   n_splits=5, use_anomaly_score=True, within_sections=False,
                   seed=1908, max_epochs=100, verbose=True, run_names=None):
    """
    Each df needs the four BASE_FEATURES columns and an integer 'label' column (0-3),
    ordered in time. Returns (fold_table, pooled_dict).
    use_anomaly_score=False gives the 4-feature LSTM (ablation), same protocol.
    within_sections=True keeps only windows inside a single state section (as in the
    thesis' main split); False keeps all windows, including ones crossing boundaries.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    runs = []
    for kind, dfs in (("none", no_plug_dfs), ("partial", partial_plug_dfs), ("plug", plug_dfs)):
        for df in dfs:
            runs.append({"type": kind, "is_no_plug": kind == "none",
                         "X4": df[BASE_FEATURES].to_numpy(np.float32),
                         "y": df["label"].to_numpy(np.int64)})
    for i, r in enumerate(runs):
        r["id"] = i
        r["name"] = run_names[i] if run_names else str(i)
    ids = np.array([r["id"] for r in runs])

    n_features = 5 if use_anomaly_score else 4
    gkf = GroupKFold(n_splits=n_splits)
    # GroupKFold is deterministic. Each run is one group, so folds get ~equal numbers of runs.
    folds = list(gkf.split(np.zeros(len(runs)), groups=ids))

    rows, all_true, all_pred = [], [], []
    for k, (tr_idx, te_idx) in enumerate(folds, start=1):
        train_runs = [runs[i] for i in tr_idx]
        test_runs = [runs[i] for i in te_idx]
        if verbose:
            print(f"\n=== Fold {k}/{n_splits}: {len(train_runs)} train runs, "
                  f"{len(test_runs)} held-out runs ({[r['name'] for r in test_runs]}) ===")

        # 1) Isolation Forest fitted on training no-plug runs only
        feats = {}
        if use_anomaly_score:
            sc, iso = fit_isolation_forest(train_runs)
            for r in runs:
                score = iso.decision_function(sc.transform(r["X4"])).astype(np.float32)
                feats[r["id"]] = np.hstack([r["X4"], score[:, None]])
        else:
            for r in runs:
                feats[r["id"]] = r["X4"]

        # 2) windows
        def build(rs):
            Xs, ys = zip(*[make_windows(feats[r["id"]], r["y"], WINDOW, within_sections)
                           for r in rs])
            return np.concatenate(Xs), np.concatenate(ys)

        fit_runs, val_runs = inner_validation_split(train_runs, seed=seed + k)
        Xtr, ytr = build(fit_runs)
        Xva, yva = build(val_runs)
        Xte, yte = build(test_runs)

        # 3) train with early stopping on the inner validation runs
        model, mu, sd, w, best_ep = train_one_fold(
            Xtr, ytr, Xva, yva, n_features, device, seed + k,
            max_epochs=max_epochs, verbose=verbose)

        # 4) evaluate once on the held-out runs
        _, pred = evaluate(model, Xte, yte, mu, sd, w, device)
        all_true.append(yte)
        all_pred.append(pred)

        share = np.bincount(yte, minlength=N_CLASSES) / len(yte)
        rows.append({
            "fold": k, "held_out_runs": len(test_runs), "windows": len(yte),
            "best_epoch": best_ep,
            "accuracy": accuracy_score(yte, pred),
            "macro_f1": f1_score(yte, pred, labels=range(N_CLASSES), average="macro", zero_division=0),
            "balanced_acc": balanced_accuracy_score(yte, pred),
            "adjacent_acc": float(np.mean(np.abs(pred - yte) <= 1)),
            "majority_share": share.max(),
            "share_0": share[0], "share_1": share[1], "share_2": share[2], "share_3": share[3],
        })
        if verbose:
            print(f"  fold {k}: acc {rows[-1]['accuracy']:.3f}, macro-F1 {rows[-1]['macro_f1']:.3f}, "
                  f"majority share {rows[-1]['majority_share']:.3f}, best epoch {best_ep}")

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    pooled = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=range(N_CLASSES), average="macro", zero_division=0),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "adjacent_acc": float(np.mean(np.abs(y_pred - y_true) <= 1)),
        "majority_baseline": np.bincount(y_true, minlength=N_CLASSES).max() / len(y_true),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=range(N_CLASSES)),
        "report": classification_report(y_true, y_pred, labels=range(N_CLASSES),
                                        digits=3, zero_division=0),
    }
    table = pd.DataFrame(rows)
    if verbose:
        print("\n=== Pooled out-of-fold results ===")
        print({k: round(v, 3) for k, v in pooled.items()
               if k not in ("confusion_matrix", "report")})
        print(pooled["report"])
        print(pooled["confusion_matrix"])
    return table, pooled

In [49]:
run_grouped_cv(no_plug_dfs, partial_plug_dfs, plug_dfs)


=== Fold 1/5: 26 train runs, 7 held-out runs (['4', '9', '14', '19', '24', '29', '32']) ===
    epoch   5  inner-val loss 0.8803
    epoch  10  inner-val loss 0.8606
  fold 1: acc 0.406, macro-F1 0.389, majority share 0.415, best epoch 2

=== Fold 2/5: 26 train runs, 7 held-out runs (['5', '10', '15', '16', '20', '25', '30']) ===
    epoch   5  inner-val loss 0.6333
    epoch  10  inner-val loss 0.6907
  fold 2: acc 0.617, macro-F1 0.432, majority share 0.525, best epoch 5

=== Fold 3/5: 26 train runs, 7 held-out runs (['0', '1', '6', '11', '21', '26', '31']) ===
    epoch   5  inner-val loss 0.9833
    epoch  10  inner-val loss 1.0236
  fold 3: acc 0.228, macro-F1 0.194, majority share 0.503, best epoch 3

=== Fold 4/5: 27 train runs, 6 held-out runs (['2', '7', '12', '17', '22', '27']) ===
    epoch   5  inner-val loss 0.9770
    epoch  10  inner-val loss 0.9927
  fold 4: acc 0.703, macro-F1 0.597, majority share 0.496, best epoch 5

=== Fold 5/5: 27 train runs, 6 held-out runs (['3

(   fold  held_out_runs  windows  best_epoch  accuracy  macro_f1  balanced_acc  \
 0     1              7    15526           2  0.406286  0.389451      0.456736   
 1     2              7    19844           5  0.617315  0.431717      0.435393   
 2     3              7    47667           3  0.228481  0.193969      0.360686   
 3     4              6    61411           5  0.703245  0.597052      0.590412   
 4     5              6    21451           1  0.827747  0.551061      0.565311   
 
    adjacent_acc  majority_share   share_0   share_1   share_2   share_3  
 0      0.779660        0.414981  0.414981  0.097900  0.320495  0.166624  
 1      0.979137        0.524693  0.524693  0.261540  0.123967  0.089800  
 2      0.774435        0.502675  0.502675  0.098076  0.381836  0.017412  
 3      0.921529        0.495660  0.495660  0.093957  0.339776  0.070606  
 4      0.946436        0.571162  0.571162  0.059904  0.309683  0.059251  ,
 {'accuracy': 0.5448616326801247,
  'macro_f1': 0.44576